ADDESTRAMENTO DISTRIBUITO CON TF.DISTRIBUTE

Trasforma il nostro codice solista in una vera orchestra passando dal calcolo su singola macchina ad un'applicazione distribuita.
TF.DISTRIBUTE server per addestrare un modello TensorFlow usando più risorse: più GPU sulla stessa macchina, più macchine, o TPU (hardware progettato per l'intelligenza artificiale creato da Google, estremamente ottimizzata per TensorFlow). Invece di far lavorare una sola GPU/CPU, TensorFlow divide il lavoro e poi ricompone i risultati

- Strategia del m irroring
- Strategia del tempo, vuoi aspettare che tutti siano pronti o correre ognuno al proprio ritmo (sincrono asincrono)
- Implementazione pratica in Keras su un singolo nodo con più GPU

Quando il modello diventa troppo grande o i dati troppo massicci per una singola scheda grafica, la soluzione risiede nella distribuzione (a volte è una necessità). La strategia di mirroring rappresenta l'approcci più comune per sfruttare li parallelismo dei dati.
In questa configurazione, ogni dispositivo riceve una copia identica del progetto originale, ma lavora su una porzione differente del batch di dati, garantendo che ogni risorsa hardware contribuisca attivamente alla discesa del gradiente. Non dividiamo il modello in pezzi, ma stiamo moltiplicando gli operatori che sanno come eseguirlo, con il concetto di data parallelismo.
Ogni CPU ha in memoria lo stesso identico modello ma guarda copie diverse, il batch globale è la somma dei batch locali

Ma come fanno le GPU ha restare sintonizzate? tramite l'algorimo All-Reduce. Se ogni GPU ha visto dati diversi tra di loro, ovviamente i calcoli ed i gradienti calcolati, sono leggermente diversi tra le varia GPU, All Recuce agisce come una riunione condominiale, raccoglie le opinioni di tutti, ne fa una media, e prende la decisione finale. La comunicazione deve essere veloce, se vogliamo che tutto funzioni, si utilizza tecnologia come NVLink, per evitare che diventi un collo di bottiglia

Il mirroring trasforma i singoli processi in un ensemble coordinato, dove la comunicazione dei gradienti funge da direttore d'orchestra per mantenere l'armonia dei pesi, e si assicura che l'armonia dei pesi rimanga perfetta.
Ma cosa succede se un 'operatore' è più lento degli altri?
Esistono due modi principali per gestire lo scambio di informazioni tra i processi: aspettare che tutti abbiano finito (sincrono) o permettere a ognuno di procedere al proprio ritmo (asincrono)
La scelta tra questi due paradigmi influenza drasticamente sia la velocità di esecuzione che la stabilità della convergenza del nostro modello Deep Learning.

PROTOCOLLI DI AGGIORNAMENTO
- L'addestramento SINCRONO prevedere che tutti i lavoratori finiscano il calcolo del gradiente prima di procedere all'aggiornamento dei pesi, garantendo determinismo.
E' matematicamente equivalente all'addestramento su singola macchina.
- L'addestramento ASINCRONO permette ai lavoratori di aggiornare i pesi non appena terminano il loro batch, evitando che i nodi più lenti rallentino l'intero di sistema. Chi finisce, aggiorna i pesi e riparte.
A differenza del modello sincrono, quello asincrono introduce rumore che può richiedere un tunning specifico
La stabiltà del gradiente mediato è il cuore della convergenza nel paradigma distribuito sincrono.

RISCHI NASCOSTI
In modalità sincrona, se una GPU è più lenta di altre (straggler) obbliga tutti i compagni a rimanere inattivi, riducendo l'efficienza, questo fenomeno è datto Straggler Problem.
In modalità asincrona, un lavoratore potrebbe aggiornare i pesi basandosi su una versione vecchia del modello, rendendo l'apprendimento meno preciso.
La riproducibilità (determinismo) degli esperimenti è molto più semplice da ottenere in configurazioni sincrone, dove l'ordine degli aggiornamenti è fisso.

Quale strategia scegliere?
Il compromesso dell'efficienza
La verità è che non esiste una soluzione universale. Se sei su una singolare macchina con 8 GPU la sincronia è quasi sempre la scelta vincente, perchè gli operatori (GPU) parlano tra di loro a velocità della luce.
L'asincronia diventa invece un'arma potente in sistemi distribuiti su larga scala, dove la latenza della rete tra server distanti renderebbe l'attesa insostenibile.

TensorFlow (tramite la Strategy) semplifica l'astrazione dell'hardware tramite le sue strategie di distribuzione.

L'oggetto STRATEGY
Interfaccia hardware-software
La classe 'MirroredStrategy' è il cuore del mirroring, quando la attivazione rileva automaticamente tutte le GPU visibili al sistema e prepara l'ambiente per la replicazione sincrona, arruolando tutte le GPU disponibili.
La creazione del modello e dell'ottimizzatore deve avvenire all'interno dello scope della strategia per attivare correttamente il mirroring dei pesi. Ogni pezzo di codice scritto all'interno dello scope diventa automaticamente replicato. Il testo del codice rimane identico a quello che scriveremmo con una sola GPU, ci sono però degli accorgimenti, tipo scalare il batch size, se una GPU gestisce 32 campioni, due GPU dovrebbero gestire 64.
TensorFlow gestisce autonomamente la distribuzione dei dati attraverso il modello model.fit, dividendo il dataset tra i dispositivi disponibili.
La strategia incapsula la logica di comunicazione rendendo il codice quasi identico a quello per singola GPU
L'uso di tf.data.Dataset è caldamente raccomandato per garantire che il caricamento dei dati non diventi il nuovo collo di bottiglia del sistema. Se però non si ottimizza la pipeline di caricamento dati dal disco utilizzando tf.data.dataset diventa lentissimo e le potenti GPU si girano i pollici aspettando che la CPU si carichi i dati.

E la competenza che permette di affrontare dataset di dimensioni reali dove è il tempo la risorsa più costosa.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

# 1. DEFINIZIONE DELLA STRATEGIA (Punto chiave 1 e 3)
# Rileva automaticamente tutte le GPU visibili
strategy = tf.distribute.MirroredStrategy()
print(f"Numero di dispositivi arruolati: {strategy.num_replicas_in_sync}") #rileva le GPU disponibili

print(tf.config.list_physical_devices())
print(tf.config.list_physical_devices('GPU'))

# 2. SCALARE IL BATCH SIZE
# Se vogliamo che ogni GPU lavori con 32 campioni (local batch),
# il batch globale deve essere 32 * numero_di_gpu (32*1).
BATCH_SIZE_PER_REPLICA = 64 #voglio che ogni batch sia di 64 campioni per ogni GPU
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
print("Global batch size:", GLOBAL_BATCH_SIZE)

# 3. PREPARAZIONE DEI DATI (tf.data raccomandato)
(train_images, train_labels), _ = datasets.mnist.load_data()
train_images = train_images.reshape(-1, 28, 28, 1).astype("float32") / 255

# Creiamo un dataset ottimizzato
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.shuffle(10000).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 4. COSTRUZIONE DEL MODELLO DENTRO LO SCOPE (Punto chiave 3)
with strategy.scope():
    # Tutto ciò che viene creato qui sarà replicato su tutte le GPU
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# 5. ADDESTRAMENTO
# Keras gestirà automaticamente la distribuzione dei batch tra le GPU
model.fit(train_dataset, epochs=5)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Numero di dispositivi arruolati: 1
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
[]
Global batch size: 64
Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9410 - loss: 0.2023
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9788 - loss: 0.0696
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.9854 - loss: 0.0471
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9891 - loss: 0.0351
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.9922 - loss: 0.0260
